# Dual-Branch Neural Benchmark For Sequence-Aware Goal Forecasting

This notebook benchmarks three models under the frozen grouped split protocol:

- `XGBoost` benchmark on `base_context`
- `XGBoost` benchmark on `base_context + trimmed sequence features`
- `DualBranchResidualMLP` with separate base and sequence branches
- `FTTransformerLite` as the fallback neural architecture if the dual-branch MLP does not improve enough

The target remains `scored_after`, and threshold tuning is handled separately from probability-model comparison.

In [1]:
from __future__ import annotations

import copy
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / "data").exists())
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts" / "sequencing_dual_branch_nn"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "scored_after"
DEVICE = torch.device("cpu")

SEQ15M_TRIMMED_FEATURES = [
    "seq15m_pressure_total",
    "seq15m_pressure_turnover_total",
    "seq15m_longest_active_streak",
    "seq15m_recency_weighted_pressure_total",
    "seq15m_active_minute_share",
    "seq15m_late_burst_count",
    "seq15m_shots_same_or_next_min_after_pressure",
]
SEQPOS_TRIMMED_FEATURES = [
    "seqpos_strict_link_rate",
    "seqpos_mean_shots_per_possession",
    "seqpos_possessions_with_shots_share",
    "seqpos_pressure_shot_escalation_share",
    "seqpos_forward_pressure_shot_share",
    "seqpos_linked_pressure_total",
    "seqpos_mean_runs_per_possession",
]
TRIMMED_SEQUENCE_FEATURES = SEQ15M_TRIMMED_FEATURES + SEQPOS_TRIMMED_FEATURES
BASE_CATEGORICAL_COLS = ["checkpoint", "position", "formation", "minute_in_window"]
BASE_BOOLEAN_COLS = ["is_home", "subbed"]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def metric_row(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    return {
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_score)) if np.unique(y_true).size > 1 else np.nan,
        "brier_score": float(brier_score_loss(y_true, y_score)),
        "target_rate": float(y_true.mean()),
    }


def tune_threshold(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float]:
    thresholds = np.linspace(0.05, 0.50, 46)
    rows = []
    for threshold in thresholds:
        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": float(
                    balanced_accuracy_score(y_true, (np.asarray(y_score) >= threshold).astype(int))
                ),
            }
        )
    table = pd.DataFrame(rows).sort_values(["balanced_accuracy", "threshold"], ascending=[False, True]).reset_index(drop=True)
    return table.iloc[0].to_dict()


def build_xgb_pipeline(feature_columns: list[str], frame: pd.DataFrame) -> Pipeline:
    numeric_cols = [column for column in feature_columns if pd.api.types.is_numeric_dtype(frame[column].dtype) and column not in BASE_BOOLEAN_COLS]
    categorical_cols = [column for column in feature_columns if column in BASE_CATEGORICAL_COLS]
    boolean_cols = [column for column in feature_columns if column in BASE_BOOLEAN_COLS]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_cols),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_cols,
            ),
            (
                "bool",
                Pipeline(
                    steps=[
                        (
                            "to_float",
                            FunctionTransformer(lambda x: x.astype(float), feature_names_out="one-to-one"),
                        ),
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                    ]
                ),
                boolean_cols,
            ),
        ]
    )
    return Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                XGBClassifier(
                    n_estimators=250,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_alpha=0.0,
                    reg_lambda=1.0,
                    min_child_weight=3,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def embedding_dim(cardinality: int) -> int:
    return min(8, max(3, (cardinality + 1) // 2))


def split_inner_indices(y: np.ndarray, frac: float = 0.15, seed: int = RANDOM_STATE) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    train_parts = []
    valid_parts = []
    for label in [0, 1]:
        indices = np.where(y == label)[0]
        rng.shuffle(indices)
        n_valid = max(1, int(round(len(indices) * frac)))
        if len(indices) <= 2:
            n_valid = 1
        valid_parts.append(indices[:n_valid])
        train_parts.append(indices[n_valid:])
    train_idx = np.sort(np.concatenate(train_parts))
    valid_idx = np.sort(np.concatenate(valid_parts))
    return train_idx, valid_idx


def prepare_fold_data(train_df: pd.DataFrame, valid_df: pd.DataFrame) -> tuple[dict[str, torch.Tensor], dict[str, torch.Tensor], dict[str, object]]:
    excluded_columns = {TARGET, "fixture_id", "fold", "partition_role", "scored_after_eval_key", "player_id", "jersey_number"}
    base_numeric_cols = [
        column
        for column in train_df.columns
        if column not in excluded_columns
        and column not in BASE_CATEGORICAL_COLS
        and column not in TRIMMED_SEQUENCE_FEATURES
        and (pd.api.types.is_numeric_dtype(train_df[column].dtype) or pd.api.types.is_bool_dtype(train_df[column].dtype))
    ]
    sequence_numeric_cols = TRIMMED_SEQUENCE_FEATURES.copy()

    numeric_medians = train_df[base_numeric_cols + sequence_numeric_cols].median()
    numeric_means = train_df[base_numeric_cols + sequence_numeric_cols].fillna(numeric_medians).mean()
    numeric_stds = train_df[base_numeric_cols + sequence_numeric_cols].fillna(numeric_medians).std(ddof=0).replace(0, 1.0)

    category_maps = {}
    cardinalities = []
    for column in BASE_CATEGORICAL_COLS:
        values = train_df[column].fillna("__missing__").astype(str)
        uniques = sorted(values.unique())
        category_maps[column] = {value: idx + 1 for idx, value in enumerate(uniques)}
        cardinalities.append(len(uniques) + 1)

    def encode_frame(frame: pd.DataFrame) -> dict[str, torch.Tensor]:
        base_numeric = (
            frame[base_numeric_cols]
            .fillna(numeric_medians[base_numeric_cols])
            .sub(numeric_means[base_numeric_cols], axis=1)
            .div(numeric_stds[base_numeric_cols], axis=1)
        )
        sequence_numeric = (
            frame[sequence_numeric_cols]
            .fillna(numeric_medians[sequence_numeric_cols])
            .sub(numeric_means[sequence_numeric_cols], axis=1)
            .div(numeric_stds[sequence_numeric_cols], axis=1)
        )
        categorical = np.column_stack(
            [
                frame[column]
                .fillna("__missing__")
                .astype(str)
                .map(category_maps[column])
                .fillna(0)
                .astype(int)
                .to_numpy()
                for column in BASE_CATEGORICAL_COLS
            ]
        )
        y = frame[TARGET].astype(int).to_numpy()
        return {
            "base_numeric": torch.tensor(base_numeric.to_numpy(dtype=np.float32), dtype=torch.float32),
            "sequence_numeric": torch.tensor(sequence_numeric.to_numpy(dtype=np.float32), dtype=torch.float32),
            "categorical": torch.tensor(categorical, dtype=torch.long),
            "y": torch.tensor(y, dtype=torch.float32),
        }

    metadata = {
        "base_numeric_cols": base_numeric_cols,
        "sequence_numeric_cols": sequence_numeric_cols,
        "categorical_cols": BASE_CATEGORICAL_COLS,
        "category_maps": category_maps,
        "cardinalities": cardinalities,
    }
    return encode_frame(train_df), encode_frame(valid_df), metadata


class ResidualBlock(nn.Module):
    def __init__(self, width: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(width, width),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(x + self.net(x))


class DualBranchResidualMLP(nn.Module):
    def __init__(self, cardinalities: list[int], base_dim: int, seq_dim: int) -> None:
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(cardinality, embedding_dim(cardinality)) for cardinality in cardinalities])
        embed_total = sum(embedding_dim(cardinality) for cardinality in cardinalities)
        self.base_proj = nn.Sequential(nn.Linear(base_dim + embed_total, 64), nn.ReLU(), nn.Dropout(0.15))
        self.base_block1 = ResidualBlock(64, 0.15)
        self.base_block2 = ResidualBlock(64, 0.10)
        self.seq_proj = nn.Sequential(nn.Linear(seq_dim, 32), nn.ReLU(), nn.Dropout(0.15))
        self.seq_block = ResidualBlock(32, 0.10)
        self.head = nn.Sequential(
            nn.Linear(96, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 1),
        )

    def forward(self, base_numeric: torch.Tensor, sequence_numeric: torch.Tensor, categorical: torch.Tensor) -> torch.Tensor:
        embedded = [embedding(categorical[:, idx]) for idx, embedding in enumerate(self.embeddings)]
        cat_vec = torch.cat(embedded, dim=1)
        base_vec = self.base_proj(torch.cat([base_numeric, cat_vec], dim=1))
        base_vec = self.base_block2(self.base_block1(base_vec))
        seq_vec = self.seq_block(self.seq_proj(sequence_numeric))
        return self.head(torch.cat([base_vec, seq_vec], dim=1)).squeeze(1)


class FTTransformerLite(nn.Module):
    def __init__(self, cardinalities: list[int], n_num_features: int, d_model: int = 32) -> None:
        super().__init__()
        self.num_weight = nn.Parameter(torch.randn(n_num_features, d_model) * 0.05)
        self.num_bias = nn.Parameter(torch.zeros(n_num_features, d_model))
        self.cat_embeddings = nn.ModuleList([nn.Embedding(cardinality, d_model) for cardinality in cardinalities])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=128,
            dropout=0.10,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 32), nn.GELU(), nn.Dropout(0.10), nn.Linear(32, 1))

    def forward(self, base_numeric: torch.Tensor, sequence_numeric: torch.Tensor, categorical: torch.Tensor) -> torch.Tensor:
        numeric = torch.cat([base_numeric, sequence_numeric], dim=1)
        num_tokens = numeric.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0)
        cat_tokens = [embedding(categorical[:, idx]).unsqueeze(1) for idx, embedding in enumerate(self.cat_embeddings)]
        cls = self.cls_token.expand(numeric.size(0), -1, -1)
        tokens = torch.cat([cls, num_tokens, *cat_tokens], dim=1)
        encoded = self.encoder(tokens)
        return self.head(encoded[:, 0, :]).squeeze(1)


def train_neural_model(model: nn.Module, train_tensors: dict[str, torch.Tensor], seed: int) -> nn.Module:
    set_seed(seed)
    train_idx, inner_valid_idx = split_inner_indices(train_tensors["y"].cpu().numpy(), frac=0.15, seed=seed)
    subtrain = {key: value[train_idx] for key, value in train_tensors.items()}
    inner_valid = {key: value[inner_valid_idx] for key, value in train_tensors.items()}

    dataset = TensorDataset(subtrain["base_numeric"], subtrain["sequence_numeric"], subtrain["categorical"], subtrain["y"])
    loader = DataLoader(dataset, batch_size=min(128, len(dataset)), shuffle=True)

    positives = max(float(subtrain["y"].sum().item()), 1.0)
    negatives = max(float(len(subtrain["y"]) - subtrain["y"].sum().item()), 1.0)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    best_state = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    patience = 12
    patience_left = patience

    for _ in range(80):
        model.train()
        for base_batch, seq_batch, cat_batch, y_batch in loader:
            base_batch = base_batch.to(DEVICE)
            seq_batch = seq_batch.to(DEVICE)
            cat_batch = cat_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(base_batch, seq_batch, cat_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            inner_logits = model(
                inner_valid["base_numeric"].to(DEVICE),
                inner_valid["sequence_numeric"].to(DEVICE),
                inner_valid["categorical"].to(DEVICE),
            )
            inner_loss = criterion(inner_logits, inner_valid["y"].to(DEVICE)).item()
        if inner_loss < best_loss - 1e-4:
            best_loss = inner_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    return model


def predict_neural_model(model: nn.Module, tensors: dict[str, torch.Tensor]) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        logits = model(
            tensors["base_numeric"].to(DEVICE),
            tensors["sequence_numeric"].to(DEVICE),
            tensors["categorical"].to(DEVICE),
        )
        return torch.sigmoid(logits).cpu().numpy()


In [2]:
sequence_df = pd.read_csv(DATA_DIR / "sequencing_dataset.csv")
row_folds = pd.read_csv(DATA_DIR / "splits" / "modeling_row_folds.csv", usecols=["fixture_id", "player_id", "checkpoint"])
row_folds["scored_after_eval_key"] = (
    row_folds["fixture_id"].astype(str) + "__" + row_folds["checkpoint"].astype(str) + "__" + row_folds["player_id"].astype(str)
)
fixture_assignments = pd.read_csv(DATA_DIR / "splits" / "fixture_assignments.csv")

sequence_df = sequence_df.merge(row_folds[["scored_after_eval_key", "fixture_id"]], on="scored_after_eval_key", how="left", validate="one_to_one")
sequence_df = sequence_df.merge(fixture_assignments, on="fixture_id", how="left", validate="many_to_one")

if sequence_df[["fixture_id", "fold", "partition_role"]].isna().any().any():
    raise ValueError("Split metadata join failed")

development_df = sequence_df[sequence_df["partition_role"] == "development"].copy().reset_index(drop=True)
fold_ids = sorted(development_df["fold"].dropna().unique())

excluded = {TARGET, "player_id", "jersey_number", "scored_after_eval_key", "fixture_id", "fold", "partition_role"}
base_context_cols = [column for column in development_df.columns if column not in excluded and not column.startswith("seq")]
trimmed_feature_cols = list(dict.fromkeys(base_context_cols + TRIMMED_SEQUENCE_FEATURES))

print("Development rows:", len(development_df))
print("Target rate:", round(float(development_df[TARGET].mean()), 4))
print("Base context columns:", len(base_context_cols))
print("Trimmed sequence columns:", len(TRIMMED_SEQUENCE_FEATURES))


Development rows: 2830
Target rate: 0.0583
Base context columns: 27
Trimmed sequence columns: 14


In [3]:
fold_rows = []
oof_parts = []

for feature_set_name, feature_columns in {
    "xgb_base_context": base_context_cols,
    "xgb_base_plus_trimmed": trimmed_feature_cols,
}.items():
    for fold_id in fold_ids:
        train_df = development_df[development_df["fold"] != fold_id].copy()
        valid_df = development_df[development_df["fold"] == fold_id].copy()
        pipeline = build_xgb_pipeline(feature_columns, train_df)
        positives = max(float(train_df[TARGET].sum()), 1.0)
        negatives = max(float(len(train_df) - train_df[TARGET].sum()), 1.0)
        pipeline.named_steps["model"].set_params(scale_pos_weight=negatives / positives)
        pipeline.fit(train_df[feature_columns], train_df[TARGET].astype(int))
        pred = pipeline.predict_proba(valid_df[feature_columns])[:, 1]
        metrics = metric_row(valid_df[TARGET].to_numpy(), pred, threshold=0.5)
        metrics.update({"model": feature_set_name, "fold": fold_id})
        fold_rows.append(metrics)
        oof_parts.append(valid_df[["scored_after_eval_key", TARGET, "fold"]].assign(model=feature_set_name, prediction=pred))

for model_name in ["dual_branch_residual_mlp", "ft_transformer_lite"]:
    for fold_index, fold_id in enumerate(fold_ids):
        train_df = development_df[development_df["fold"] != fold_id].copy()
        valid_df = development_df[development_df["fold"] == fold_id].copy()
        train_tensors, valid_tensors, metadata = prepare_fold_data(train_df, valid_df)
        if model_name == "dual_branch_residual_mlp":
            model = DualBranchResidualMLP(metadata["cardinalities"], train_tensors["base_numeric"].shape[1], train_tensors["sequence_numeric"].shape[1])
        else:
            model = FTTransformerLite(metadata["cardinalities"], train_tensors["base_numeric"].shape[1] + train_tensors["sequence_numeric"].shape[1])
        model = model.to(DEVICE)
        model = train_neural_model(model, train_tensors, seed=RANDOM_STATE + fold_index)
        pred = predict_neural_model(model, valid_tensors)
        metrics = metric_row(valid_df[TARGET].to_numpy(), pred, threshold=0.5)
        metrics.update({"model": model_name, "fold": fold_id})
        fold_rows.append(metrics)
        oof_parts.append(valid_df[["scored_after_eval_key", TARGET, "fold"]].assign(model=model_name, prediction=pred))

fold_metrics_df = pd.DataFrame(fold_rows).sort_values(["model", "fold"]).reset_index(drop=True)
oof_df = pd.concat(oof_parts, ignore_index=True)

summary_rows = []
threshold_rows = []
for model_name in sorted(fold_metrics_df["model"].unique()):
    model_fold_df = fold_metrics_df[fold_metrics_df["model"] == model_name].copy()
    model_oof_df = oof_df[oof_df["model"] == model_name].copy()
    best_threshold = tune_threshold(model_oof_df[TARGET].to_numpy(), model_oof_df["prediction"].to_numpy())
    threshold_rows.append({"model": model_name, **best_threshold})
    tuned_metrics = metric_row(model_oof_df[TARGET].to_numpy(), model_oof_df["prediction"].to_numpy(), threshold=best_threshold["threshold"])
    summary_rows.append(
        {
            "model": model_name,
            "mean_pr_auc": float(model_fold_df["pr_auc"].mean()),
            "std_pr_auc": float(model_fold_df["pr_auc"].std(ddof=0)),
            "mean_balanced_accuracy_at_0_5": float(model_fold_df["balanced_accuracy"].mean()),
            "mean_roc_auc": float(model_fold_df["roc_auc"].mean()),
            "mean_brier_score": float(model_fold_df["brier_score"].mean()),
            "tuned_threshold": float(best_threshold["threshold"]),
            "oof_balanced_accuracy_at_tuned_threshold": float(tuned_metrics["balanced_accuracy"]),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values(["mean_pr_auc", "oof_balanced_accuracy_at_tuned_threshold"], ascending=[False, False]).reset_index(drop=True)
threshold_df = pd.DataFrame(threshold_rows).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

fold_metrics_df.to_csv(ARTIFACT_DIR / "fold_metrics.csv", index=False)
oof_df.to_csv(ARTIFACT_DIR / "oof_predictions.csv", index=False)
summary_df.to_csv(ARTIFACT_DIR / "model_summary.csv", index=False)
threshold_df.to_csv(ARTIFACT_DIR / "threshold_summary.csv", index=False)
pd.DataFrame({"feature": TRIMMED_SEQUENCE_FEATURES}).to_csv(ARTIFACT_DIR / "trimmed_sequence_features.csv", index=False)

summary_df


,model,mean_pr_auc,std_pr_auc,mean_balanced_accuracy_at_0_5,mean_roc_auc,mean_brier_score,tuned_threshold,oof_balanced_accuracy_at_tuned_threshold
0,ft_transformer_lite,0.121022,0.040310,0.572438,0.625264,0.244202,0.50,0.573114
1,xgb_base_context,0.105957,0.024842,0.558691,0.629986,0.119368,0.21,0.602138
2,xgb_base_plus_trimmed,0.103924,0.022159,0.550704,0.612747,0.101784,0.07,0.591273
3,dual_branch_residual_mlp,0.101869,0.017179,0.612654,0.643988,0.237690,0.49,0.618205
